# Colab SSH Setup

Run this notebook in Colab to get SSH access. Then control everything from your local Cursor terminal.

**Steps:**
1. Run all cells below
2. Copy the SSH command printed at the end
3. Paste it in your local terminal (Cursor)
4. Run your scripts on Colab's GPU from home

In [ ]:
# Step 1: Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Step 2: Mount Google Drive (persists models/data across sessions)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3: Install dependencies
!pip install -q \
    "numpy>=1.26,<2" \
    diffusers==0.30.0 \
    transformers==4.44.0 \
    accelerate==0.33.0 \
    peft==0.12.0 \
    bitsandbytes==0.43.3 \
    datasets==2.20.0 \
    huggingface_hub==0.24.5 \
    controlnet_aux==0.0.9 \
    xformers \
    einops \
    tomesd \
    "imageio[ffmpeg]" \
    opencv-python-headless \
    gradio==4.42.0 \
    safetensors \
    sentencepiece \
    protobuf
print("Dependencies installed.")

In [ ]:
# Step 4: Clone the repo and set up project structure
import os

DRIVE_ROOT = "/content/drive/MyDrive/ImgGen"
PROJECT = "/content/ImgGen"

# Clone repo if not already there
if not os.path.exists(PROJECT):
    !git clone https://github.com/Mehul159/ImgGen.git /content/ImgGen
else:
    !cd /content/ImgGen && git pull

# Symlink heavy dirs to Google Drive
for folder in ["models", "lora_weights", "data"]:
    drive_path = os.path.join(DRIVE_ROOT, folder)
    local_path = os.path.join(PROJECT, folder)
    os.makedirs(drive_path, exist_ok=True)
    if os.path.isdir(local_path) and not os.path.islink(local_path):
        !rm -rf {local_path}
    if not os.path.exists(local_path):
        os.symlink(drive_path, local_path)
        print(f"  {folder}/ -> Drive")

os.makedirs(os.path.join(PROJECT, "outputs"), exist_ok=True)
print(f"\nProject ready at {PROJECT}")

In [ ]:
# Step 5: Set up SSH with cloudflared tunnel
import secrets
password = secrets.token_urlsafe(12)
print(f"Root password: {password}")

import os
os.environ['COLAB_SSH_PASSWORD'] = password

In [ ]:
%%bash -s "$COLAB_SSH_PASSWORD"
PASSWORD=$1

# Set password
echo "root:$PASSWORD" | chpasswd

# Install & configure SSH
apt-get -qq update && apt-get -qq install -y openssh-server > /dev/null 2>&1
mkdir -p /var/run/sshd
sed -i 's/#PermitRootLogin.*/PermitRootLogin yes/' /etc/ssh/sshd_config
sed -i 's/#PasswordAuthentication.*/PasswordAuthentication yes/' /etc/ssh/sshd_config
service ssh restart
echo "--- SSH server running ---"

# Install cloudflared
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
rm -f cloudflared-linux-amd64.deb
echo "--- cloudflared installed ---"

# Start tunnel and wait for URL
nohup cloudflared tunnel --url ssh://localhost:22 --no-autoupdate > /tmp/cf.log 2>&1 &
echo "--- Waiting for tunnel URL ---"

for i in $(seq 1 30); do
    sleep 2
    URL=$(grep -oP 'https://[\w-]+\.trycloudflare\.com' /tmp/cf.log 2>/dev/null | head -1)
    if [ -n "$URL" ]; then
        HOSTNAME=$(echo $URL | sed 's|https://||')
        echo ""
        echo "============================================================"
        echo "  SSH TUNNEL READY"
        echo "============================================================"
        echo ""
        echo "  Password: $PASSWORD"
        echo ""
        echo "  CONNECT FROM YOUR LOCAL TERMINAL:"
        echo ""
        echo "  ssh -o ProxyCommand=\"cloudflared access ssh --hostname $HOSTNAME\" root@$HOSTNAME"
        echo ""
        echo "  Then: cd /content/ImgGen && python preprocess.py"
        echo "============================================================"
        exit 0
    fi
    printf "."
done

echo ""
echo "Failed. Log:"
cat /tmp/cf.log

In [ ]:
# Step 6: Keep alive — run this cell and leave it running
import time
print("Keeping session alive... (leave this running)")
print("SSH in from your local terminal using the details above.")
while True:
    time.sleep(60)
    print(".", end="", flush=True)